# 05 · Interface « Derja Voice Lab » : lier les notebooks à une démo interactive

Ce notebook relie les modèles entraînés dans les notebooks 01 et 02 à une interface web (Gradio) :
on parle en tunisien (micro) ou on charge un fichier, on choisit la langue de traduction et la méthode,
et l'interface montre **chaque étape** que traverse la voix :

| # | Étape | Ce qu'on voit |
|---|---|---|
| 1 | Signal d'entrée | forme d'onde, fréquence d'origine → 16 kHz, niveau, saturation |
| 2 | Détection de parole | zones de parole, découpage en morceaux ≤ 25 s |
| 3 | Représentation acoustique | spectrogramme log-Mel que lit le modèle |
| 4 | Reconnaissance (ASR) | texte tunisien, confiance par token (Whisper) ou émissions par trame (CTC) |
| 5 | Code-switching | mots colorés par script, type TUN/MIXED/FOR, indice CMI |
| 6 | Normalisation | chaque règle appliquée, avec le diff avant/après |
| 7 | Post-correction LLM (option) | diff des corrections du LLM |
| 8 | Traduction | directe, pipeline (NLLB ou LLM) ou les deux, avec leur accord chrF |
| 9 | Résultat | temps passé dans chaque étape |

**Lien notebook ↔ interface** : les notebooks enregistrent leurs meilleurs modèles dans
`workdir/models/asr_best`, `ast_best` et `mt_best` ; l'interface les charge automatiquement. Ses tableaux
« Résultats des expériences » lisent `results/*.csv`, écrits par les notebooks.

In [ ]:
import os, subprocess, sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/roudayn-kouka/tedxtn-tunisian-speech.git"
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ.setdefault("TEDXTN_WORKDIR", "/content/drive/MyDrive/tedxtn")
    project = Path("/content/drive/MyDrive/tedxtn-tunisian-speech")
    if not project.exists():
        project = Path("/content/tedxtn-tunisian-speech")
        if not project.exists():
            subprocess.run(["git", "clone", "-q", REPO_URL, str(project)], check=True)
    os.chdir(project)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

## 1. Vérifier les modèles disponibles
Si un modèle fine-tuné manque, l'interface se rabat sur le modèle de base et l'indique dans l'étape concernée
(les résultats sont alors bien moins bons : Whisper zero-shot dépasse 90 % de WER sur TEDxTN).

In [ ]:
import pandas as pd
from IPython.display import display
from common.config import AppModels, MODELS_DIR, RESULTS_DIR
m = AppModels()
pd.DataFrame([{"rôle": r, "chemin": p, "présent": Path(p).exists(), "repli": f}
              for r, p, f in [("ASR", m.asr, m.asr_fallback), ("Traduction directe", m.ast_e2e, m.ast_e2e_fallback),
                              ("Traduction texte (NLLB)", m.mt, m.mt_fallback)]])

Pour utiliser d'autres modèles (par exemple ceux poussés sur le Hub par l'équipe), définir les variables
d'environnement avant de construire le pipeline :
```python
os.environ["TEDXTN_ASR_MODEL"] = "votre-compte/whisper-small-tedxtn"
os.environ["TEDXTN_USE_LLM"] = "1"
```

## 2. Tester le pipeline sans l'interface
On fait passer quelques segments du test dans le pipeline complet et on compare aux références.

In [ ]:
from common import data, metrics
from common.pipeline import TunisianSpeechPipeline
pipe = TunisianSpeechPipeline(m)
segs = sorted((Path(os.environ.get("TEDXTN_WORKDIR", "workdir")) / "audio_segments").glob("test__*.wav"))[:5]
if segs:
    df = data.load_corpus(download=False).set_index("seg_id")
    rows = []
    for p in segs:
        res = pipe.run(str(p), target="Anglais", mode="compare")
        ref = df.loc[p.stem]
        rows.append({"référence": ref.transcription, "ASR": res.transcript,
                     "WER": metrics.segment_wer(ref.transcription, res.transcript),
                     "réf. anglaise": ref.translation, **res.translations,
                     "temps (s)": round(res.total_seconds, 2)})
    display(pd.DataFrame(rows))
else:
    print("Aucun segment de test trouvé (exécuter la section 4 du notebook 01).")

## 3. Lancer l'interface
Sur Colab, `share=True` crée un lien public `https://….gradio.live` valable 72 h, à ouvrir sur n'importe quel
appareil (le micro du téléphone fonctionne). En local, ouvrir http://localhost:7860.
La cellule reste « en cours d'exécution » tant que l'interface tourne ; l'interrompre arrête le serveur.

In [ ]:
from app.app import launch
demo = launch(share=IN_COLAB, pipeline=pipe, debug=False)

## 4. Conseils pour la démonstration
- Parler près du micro, dans un endroit calme ; des phrases de 5 à 20 s donnent les meilleurs résultats
  (proches de la durée des segments d'entraînement).
- Mode **Comparer les deux** pour montrer la différence entre traduction directe et pipeline.
- Montrer l'étape 4 : les tokens en rouge sont ceux dont le modèle est le moins sûr, souvent les mots
  français ou les noms propres ; l'étape 5 montre justement ces mots en code-switching.
- Pour le français et l'arabe standard, rappeler qu'aucune référence n'existe dans TEDxTN : ces traductions
  n'ont pas été évaluées automatiquement.